# Propensity-score diagnostics

Run the full diagnostics suite — positivity, weight distributions/ESS, and covariate balance — on a synthetic dataset with known ground truth, then visualize balance (Love plot) and the IPW weight distribution.

In [ ]:
import matplotlib.pyplot as plt

from CausalEstimate import run_diagnostics
from CausalEstimate.datasets import load_binary_with_probas
from CausalEstimate.diagnostics import compute_balance_table, check_balance
from CausalEstimate.vis.plotting import plot_love, plot_ps_boxplot, plot_weight_dist, plot_zipper

df = load_binary_with_probas(n_samples=5000, random_state=42)
# derive a few extra covariates so the balance table is more interesting
df["X1*X2"] = df["X1"] * df["X2"]
df["X1^2"] = df["X1"] ** 2
df["X1>0"] = (df["X1"] > 0).astype(float)
df["X2^2"] = df["X2"] ** 2

report = run_diagnostics(df, ps_col="ps", treatment_col="treatment", covariate_cols=["X1", "X2", "X1*X2", "X1^2", "X1>0", "X2^2"])
print(report["flags"])
print(report["positivity"])
print(report["weights"])
report["balance"]

In [ ]:
table = compute_balance_table(df, covariate_cols=["X1", "X2", "X1*X2", "X1^2", "X1>0", "X2^2"])
print(check_balance(table))

fig, ax = plot_love(table)
plt.show()

fig, ax = plot_weight_dist(df)
plt.show()

Propensity-score boxplots before and after weighting: under good weighting the treated and control boxes should nearly coincide.

In [ ]:
fig, ax = plot_ps_boxplot(df)
plt.show()

Zipper plot: bootstrap confidence intervals from repeated simulations, colored by whether they cover the true ATE. Nominal 95% intervals should miss about 5% of the time.

In [ ]:
import numpy as np

from CausalEstimate.core.multi_estimator import MultiEstimator
from CausalEstimate.estimators import IPW

ipw = IPW(effect_type="ATE", treatment_col="treatment", outcome_col="Y", ps_col="ps")
truth, lower, upper = [], [], []
for seed in range(100):
    d, params = load_binary_with_probas(n_samples=2000, random_state=seed, return_params=True)
    res = MultiEstimator([ipw]).compute_effects(d, n_bootstraps=200)["IPW"]
    truth.append(params["true_ate"])
    lower.append(res["CI95_lower"])
    upper.append(res["CI95_upper"])

fig, ax = plot_zipper(np.array(truth), lower, upper)
plt.show()